In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter
import contextlib
from dataclasses import dataclass

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/economicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def dummy(input: str=None) -> list:
    return {'key1': 1, 'key2': 2}

In [11]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{MY_DATABASE_FILE}' AS project")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        for tab in ['project.edge_list_combined']: 
            self.db.sql(f"DROP TABLE IF EXISTS {tab}")

        with contextlib.suppress(Exception):
            self.db.create_function('dummy', 
                                        dummy, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )

        self.db.sql("CREATE OR REPLACE TABLE memory.t1 AS SELECT 42 AS i, 84 AS j")    
        self.db.sql("SHOW ALL TABLES").show()          
        print(self.db.sql("SHOW ALL TABLES").df())
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
# s = SetUp()

┌──────────┬─────────┬─────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────